In [1]:
!pip install --upgrade transformers sentencepiece -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
PyTorch version: 2.4.1+cu124
CUDA available: True


In [2]:
DATA_DIR = ''
OUTPUT_DIR = ''

from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import json
import numpy as np
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
from dataclasses import dataclass
import time
import networkx as nx
from collections import defaultdict
import os

import faiss
print(f"✓ FAISS version: {faiss.__version__}")
print(f"  NumPy version: {np.__version__}")
print(f"\nPyTorch CUDA available: {torch.cuda.is_available()}")

✓ FAISS version: 1.13.2
  NumPy version: 1.26.3

PyTorch CUDA available: True


In [5]:
# Model name
MODEL_NAME = "e5-intertext"
GRAPH_PATH   = "bdc_graph.graphml"
N_NEIGHBOURS = 5     # neighbourhood size — tune on validation set
K0_RRF       = 60    # RRF constant (Cormack)
K_PROXY      = 20    # top-k retrieved per proxy query before fusion


# Retrieval configuration
K_VALUES = [5, 10, 20]  # Top-k values to retrieve

In [9]:
# check graph file
print('Loading epistolary graph...')
G = nx.read_graphml(GRAPH_PATH)
print(f'  Nodes : {G.number_of_nodes():,}')
print(f'  Edges : {G.number_of_edges():,}')

Loading epistolary graph...
  Nodes : 13,114
  Edges : 42,458


In [15]:
def load_embeddings(embeddings_dir: str, model_name: str) -> Tuple[np.ndarray, np.ndarray, List[str], List[str], Dict]:
    """
    Load embeddings and chunk IDs.

    Returns:
        (bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata)
    """
    print("Loading embeddings and IDs...\n")

    # Load embeddings
    bdc_emb_path = f"bdc_embeddings_{model_name}.npy"
    psc_emb_path = f"psc_embeddings_{model_name}.npy"

    print(f"Loading BDC embeddings from: {Path(bdc_emb_path).name}")
    bdc_embeddings = np.load(bdc_emb_path)
    print(f"  Shape: {bdc_embeddings.shape}")

    print(f"Loading PSC embeddings from: {Path(psc_emb_path).name}")
    psc_embeddings = np.load(psc_emb_path)
    print(f"  Shape: {psc_embeddings.shape}\n")

    # Load IDs from JSON
    bdc_ids_path = f"bdc_ids_{model_name}.json"
    psc_ids_path = f"psc_ids_{model_name}.json"

    print(f"Loading BDC IDs from: {Path(bdc_ids_path).name}")
    with open(bdc_ids_path, 'r') as f:
        bdc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(bdc_ids):,} IDs")

    print(f"Loading PSC IDs from: {Path(psc_ids_path).name}")
    with open(psc_ids_path, 'r') as f:
        psc_ids = json.load(f)  # Load entire JSON array
    print(f"  Loaded {len(psc_ids):,} IDs\n")

    # Load metadata
    metadata_path = f"metadata_{model_name}.json"
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    print(f"✓ Loaded metadata\n")

    # Verification
    assert bdc_embeddings.shape[0] == len(bdc_ids), "BDC embeddings and IDs mismatch!"
    assert psc_embeddings.shape[0] == len(psc_ids), "PSC embeddings and IDs mismatch!"
    assert bdc_embeddings.shape[1] == psc_embeddings.shape[1], "Embedding dimensions mismatch!"

    return bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata


# Load data
bdc_embeddings, psc_embeddings, bdc_ids, psc_ids, metadata = load_embeddings(
    DATA_DIR,
    MODEL_NAME
)

print(f"\nLoaded:")
print(f"  BDC queries: {len(bdc_ids):,}")
print(f"  PSC candidates: {len(psc_ids):,}")
print(f"  Embedding dimension: {bdc_embeddings.shape[1]}")

Loading embeddings and IDs...

Loading BDC embeddings from: bdc_embeddings_e5-intertext.npy
  Shape: (961431, 1024)
Loading PSC embeddings from: psc_embeddings_e5-intertext.npy
  Shape: (46408, 1024)

Loading BDC IDs from: bdc_ids_e5-intertext.json
  Loaded 961,431 IDs
Loading PSC IDs from: psc_ids_e5-intertext.json
  Loaded 46,408 IDs

✓ Loaded metadata


Loaded:
  BDC queries: 961,431
  PSC candidates: 46,408
  Embedding dimension: 1024


In [16]:
# Get embedding dimension
embedding_dim = psc_embeddings.shape[1]

print(f"PSC embeddings shape: {psc_embeddings.shape}")
print(f"Embedding dimension: {embedding_dim}")
print()

# Build FAISS index (IndexFlatIP)
print("Building FAISS index...")
index = faiss.IndexFlatIP(embedding_dim)

# Add PSC embeddings to index
index.add(psc_embeddings.astype('float32'))

print(f"✓ FAISS index built")
print(f"  Index type: IndexFlatIP (exact inner product)")
print(f"  Total vectors: {index.ntotal:,}")
print(f"  Dimension: {embedding_dim}")

PSC embeddings shape: (46408, 1024)
Embedding dimension: 1024

Building FAISS index...
✓ FAISS index built
  Index type: IndexFlatIP (exact inner product)
  Total vectors: 46,408
  Dimension: 1024


In [17]:
# Create mappings
bdc_id_to_idx = {chunk_id: idx for idx, chunk_id in enumerate(bdc_ids)}
psc_id_to_idx = {chunk_id: idx for idx, chunk_id in enumerate(psc_ids)}

print(f"BDC mapping: {len(bdc_id_to_idx):,} queries")
print(f"PSC mapping: {len(psc_id_to_idx):,} candidates")

# Verify mappings work
sample_bdc_id = bdc_ids[0]
sample_psc_id = psc_ids[0]

print(f"\nSample BDC ID: {sample_bdc_id} → index {bdc_id_to_idx[sample_bdc_id]}")
print(f"Sample PSC ID: {sample_psc_id} → index {psc_id_to_idx[sample_psc_id]}")

BDC mapping: 961,431 queries
PSC mapping: 46,408 candidates

Sample BDC ID: 10015_sent_0 → index 0
Sample PSC ID: 001_Tertullianus_Apologeticus-adversus-gentes_window_0 → index 0


In [18]:
# Parameters
MAX_K = max(K_VALUES)

# Build retrieval results: {query_id: [(candidate_id, score), ...]}
retrieval_results = {}

print(f"Retrieving top-{MAX_K} candidates for each query...")
print(f"Total queries to process: {len(bdc_ids):,}\n")

for query_id in tqdm(bdc_ids, desc="Retrieving"):
    # Get query index
    query_idx = bdc_id_to_idx[query_id]

    # Search FAISS index
    distances, indices = index.search(
        bdc_embeddings[query_idx:query_idx+1],
        MAX_K
    )

    # Convert to (candidate_id, similarity_score) tuples
    candidates = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), 1):
        candidate_id = psc_ids[idx]
        similarity = float(dist)  # cosine similarity from normalized vectors
        candidates.append((candidate_id, similarity))

    retrieval_results[query_id] = candidates

print(f"\n✓ Retrieval complete")
print(f"  Total queries: {len(retrieval_results):,}")
print(f"  Candidates per query: {MAX_K}")

# Verify structure
sample_query = list(retrieval_results.keys())[0]
print(f"\nSample retrieval result:")
print(f"  Query ID: {sample_query}")
print(f"  Top 3 candidates:")
for cand_id, score in retrieval_results[sample_query][:3]:
    print(f"    - {cand_id}: {score:.4f}")

Retrieving top-20 candidates for each query...
Total queries to process: 961,431



Retrieving:   0%|          | 0/961431 [00:00<?, ?it/s]


✓ Retrieval complete
  Total queries: 961,431
  Candidates per query: 20

Sample retrieval result:
  Query ID: 10015_sent_0
  Top 3 candidates:
    - 033_Augustinus-Hipponensis_Epistolae_window_3477: 0.7647
    - 022_Hieronymus-Stridonensis_Epistolae_window_1738: 0.7571
    - 066_Benedictus-Nursiae_Regula-cum-commentariis_window_3169: 0.7568


In [20]:
# Create results for each k value (for compatibility with evaluation)
results_by_k = {}

for k in K_VALUES:
    results_by_k[k] = {
        query_id: candidates[:k]
        for query_id, candidates in retrieval_results.items()
    }

# Save retrieval results
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "total_queries": len(retrieval_results),
        "total_candidates": len(psc_ids),
        "max_k": MAX_K,
        "k_values": K_VALUES,
        "index_type": "FAISS IndexFlatIP",
    },
    "retrieval_results": {
        query_id: [
            {
                "candidate_id": cand_id,
                "similarity": score,
                "rank": rank
            }
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in retrieval_results.items()
    }
}

# Convert OUTPUT_DIR to Path if it's a string
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
output_path = "retrieval_results_{MODEL_NAME.lower()}.json"
with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f"Saved full retrieval results to: {output_path}")

#  save PKL
import pickle
pickle_path = "retrieval_results_{MODEL_NAME.lower()}.pkl"
with open(pickle_path, 'wb') as f:
    pickle.dump(retrieval_results, f)

print(f"\nFiles saved:")
print(f"  - JSON: {output_path}")
print(f"  - Pickle: {pickle_path}")

Saved full retrieval results to: retrieval_results_{MODEL_NAME.lower()}.json

Files saved:
  - JSON: retrieval_results_{MODEL_NAME.lower()}.json
  - Pickle: retrieval_results_{MODEL_NAME.lower()}.pkl


In [22]:
# --- Build letter -> chunk indices mapping ---
# chunk_id format: '{letter_id}_sent_{n}' or '{letter_id}_sent_{n}_{m}'
# letter_id in chunk = numeric string (e.g. '10015')
# graph node id = 'file10015'  →  strip 'file' prefix for matching

letter_to_chunk_indices = defaultdict(list)  # numeric letter_id -> [embedding row indices]
for idx, chunk_id in enumerate(bdc_ids):
    letter_id = chunk_id.split('_')[0]   # '10015_sent_188_190' -> '10015'
    letter_to_chunk_indices[letter_id].append(idx)

print(f'\nLetter->chunk index built: {len(letter_to_chunk_indices):,} letters')
sample_lid = list(letter_to_chunk_indices.keys())[0]
print(f'  Sample: letter {sample_lid} -> {len(letter_to_chunk_indices[sample_lid])} chunks')


Letter->chunk index built: 11,858 letters
  Sample: letter 10015 -> 520 chunks


In [34]:
# --- MRR functions ---

def get_proxy_embedding(letter_id_numeric: str) -> np.ndarray | None:
    """
    Compute proxy query vector for a neighbour letter:
    mean of all its BDC chunk embeddings.
    Returns None if letter has no chunks in the embedding matrix.
    """
    indices = letter_to_chunk_indices.get(letter_id_numeric)
    if not indices:
        return None
    vecs = bdc_embeddings[indices]          # (n_chunks, dim)
    mean_vec = vecs.mean(axis=0)            # (dim,)
    norm = np.linalg.norm(mean_vec)
    if norm == 0:
        return None
    return (mean_vec / norm).astype('float32')   # re-normalise after averaging


def faiss_query(query_vec: np.ndarray, k: int) -> list[tuple[str, float]]:
    """
    Query FAISS index with a single vector.
    Returns [(candidate_id, score), ...] of length k.
    """
    distances, indices = index.search(query_vec.reshape(1, -1), k)
    return [(psc_ids[idx], float(dist))
            for dist, idx in zip(distances[0], indices[0])
            if idx >= 0]


def reciprocal_rank_fusion(
    ranked_lists: list[list[tuple[str, float]]],
    k0: int = 60
) -> list[tuple[str, float]]:
    """
    Merge ranked lists via RRF (Cormack et al. 2009).
    RRF(d) = sum_r 1 / (k0 + rank_r(d))

    Parameters
    ----------
    ranked_lists : list of [(candidate_id, score), ...] — score unused in RRF
    k0           : RRF constant (default 60)

    Returns
    -------
    [(candidate_id, rrf_score), ...] sorted descending by rrf_score
    """
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, (cand_id, _) in enumerate(ranked, start=1):
            scores[cand_id] += 1.0 / (k0 + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def enrich_metadata_retrieve(
    query_chunk_id: str,
    n_neighbours: int = N_NEIGHBOURS,
    k_proxy: int = K_PROXY,
    k0: int = K0_RRF
) -> tuple[list[tuple[str, float]], int]:
    """
    Graph-neighbourhood retrieval + RRF for a single query chunk.

    Returns
    -------
    (fused_ranked_list, n_proxy_lists_used)
    fused_ranked_list : [(candidate_id, rrf_score), ...]
    n_proxy_lists_used: number of neighbour proxy lists added (0 = isolated fallback)
    """
    # Baseline list for this query chunk (already computed)
    baseline = retrieval_results.get(query_chunk_id, [])

    # Identify query letter
    query_letter_numeric = query_chunk_id.split('_')[0]   # '10015'
    graph_node_id = f'file{query_letter_numeric}'          # 'file10015'

    # Get neighbourhood from graph
    if graph_node_id not in G or G.degree(graph_node_id) == 0:
        # Isolated node: return baseline unchanged
        return [(cid, score) for cid, score in baseline], 0

    neighbours = list(G[graph_node_id].items())
    neighbours.sort(key=lambda x: x[1].get('date_min', ''))
    neighbours = neighbours[:n_neighbours]

    # Build proxy ranked lists from neighbours
    proxy_lists = []
    for nbr_node_id, _ in neighbours:
        nbr_letter_numeric = nbr_node_id.replace('file', '')
        proxy_vec = get_proxy_embedding(nbr_letter_numeric)
        if proxy_vec is None:
            continue
        proxy_results = faiss_query(proxy_vec, k_proxy)
        proxy_lists.append(proxy_results)

    if not proxy_lists:
        return [(cid, score) for cid, score in baseline], 0

    # Fuse: baseline list + all proxy lists
    all_lists = [baseline] + proxy_lists
    fused = reciprocal_rank_fusion(all_lists, k0=k0)
    return fused, len(proxy_lists)

In [38]:
# --- Modified retrieval with similarity threshold on proxy chunks ---

def get_proxy_embedding_filtered(letter_id_numeric: str, 
                                  query_vec: np.ndarray,
                                  sim_threshold: float = 0.0) -> np.ndarray | None:
    """
    Proxy embedding using only chunks of the neighbour letter that score
    above sim_threshold against the query vector.
    Falls back to mean of all chunks if none pass the threshold.
    Returns None if letter has no chunks.
    """
    indices = letter_to_chunk_indices.get(letter_id_numeric)
    if not indices:
        return None
    vecs = bdc_embeddings[indices]                    # (n_chunks, dim)
    
    if sim_threshold > 0.0:
        # cosine similarity against query (vectors already normalised)
        sims = vecs @ query_vec                       # (n_chunks,)
        mask = sims >= sim_threshold
        if mask.sum() > 0:
            vecs = vecs[mask]
        else:
            return None  # drop this neighbour entirely, don't fall back
        # else: no chunk passes threshold → fall back to all chunks
    
    mean_vec = vecs.mean(axis=0)
    norm = np.linalg.norm(mean_vec)
    if norm == 0:
        return None
    return (mean_vec / norm).astype('float32')


def enrich_metadata_retrieve_filtered(
    query_chunk_id: str,
    n_neighbours: int = N_NEIGHBOURS,
    k_proxy: int = K_PROXY,
    k0: int = K0_RRF,
    sim_threshold: float = 0.0
) -> tuple[list[tuple[str, float]], int]:
    """
    Graph-neighbourhood retrieval + RRF with optional similarity threshold
    on proxy chunk selection.
    """
    baseline = retrieval_results.get(query_chunk_id, [])
    query_letter_numeric = query_chunk_id.split('_')[0]
    graph_node_id = f'file{query_letter_numeric}'

    if graph_node_id not in G or G.degree(graph_node_id) == 0:
        return [(cid, score) for cid, score in baseline], 0

    # query vector for threshold comparison
    query_idx = bdc_id_to_idx.get(query_chunk_id)
    query_vec = bdc_embeddings[query_idx].astype('float32') if query_idx is not None else None

    neighbours = sorted(G[graph_node_id].items(),
                        key=lambda x: x[1].get('date_min', ''))[:n_neighbours]

    proxy_lists = []
    for nbr_node_id, _ in neighbours:
        nbr_letter_numeric = nbr_node_id.replace('file', '')
        if query_vec is not None and sim_threshold > 0.0:
            proxy_vec = get_proxy_embedding_filtered(
                nbr_letter_numeric, query_vec, sim_threshold
            )
        else:
            proxy_vec = get_proxy_embedding(nbr_letter_numeric)
        if proxy_vec is None:
            continue
        proxy_lists.append(faiss_query(proxy_vec, k_proxy))

    if not proxy_lists:
        return [(cid, score) for cid, score in baseline], 0

    fused = reciprocal_rank_fusion([baseline] + proxy_lists, k0=k0)
    return fused, len(proxy_lists)

In [54]:
# --- Check GT chunks for letter 10041 specifically ---

gt_test = {
    # letter 10404 — implicit, Augustine
    '10506_sent_27':     ['040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140'],
}

THRESHOLDS = [0.0, 0.78]

for sample_chunk, relevant in gt_test.items():
    if sample_chunk not in bdc_id_to_idx:
        print(f'{sample_chunk}: NOT IN EMBEDDING MATRIX — skipping\n')
        continue

    letter_id = sample_chunk.split('_')[0]
    query_idx = bdc_id_to_idx[sample_chunk]
    query_vec = bdc_embeddings[query_idx].astype('float32')

    print(f'{"═"*80}')
    print(f'Query    : {sample_chunk}')
    print(f'Relevant : {relevant[0]}')
    nd = G.nodes.get(f'file{letter_id}', {})
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→'
          f'{nd.get("recipient_ref","?")}  {nd.get("date_str","?")}  '
          f'degree={G.degree(f"file{letter_id}")}')
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→{nd.get("recipient_ref","?")}  '
          f'{nd.get("date_str","?")}  degree={G.degree(f"file{letter_id}")}')

    # neighbour sim summary
    graph_node = f'file{letter_id}'
    if graph_node in G and G.degree(graph_node) > 0:
        neighbours = sorted(G[graph_node].items(),
                            key=lambda x: x[1].get('date_min', ''))[:N_NEIGHBOURS]
        print(f'\nNeighbour chunk similarity to query:')
        for nbr_node, edata in neighbours:
            nbr_num = nbr_node.replace('file', '')
            indices = letter_to_chunk_indices.get(nbr_num, [])
            if not indices:
                continue
            vecs = bdc_embeddings[indices]
            sims = vecs @ query_vec
            passing = {t: int((sims >= t).sum()) for t in THRESHOLDS if t > 0}
            passing_str = '  '.join(f't={t}: {v}/{len(indices)}' for t, v in passing.items())
            nnd = G.nodes[nbr_node]
            print(f'  {nbr_node} ({nnd.get("sender_ref","?")}→{nnd.get("recipient_ref","?")}, '
                  f'{edata.get("date_min","?")})'
                  f'  sim=[{sims.min():.3f},{sims.max():.3f}]  {passing_str}')

    # run all variants
    results_by_thresh = {}
    for t in THRESHOLDS:
        fused, n_proxy = enrich_metadata_retrieve_filtered(sample_chunk, sim_threshold=t)
        results_by_thresh[t] = ([cid for cid, _ in fused[:10]], n_proxy)
    baseline_ids = [cid for cid, _ in retrieval_results[sample_chunk][:10]]

    # collect candidates — put relevant first so it's always visible
    all_ids = list(dict.fromkeys(
        relevant +
        baseline_ids +
        [cid for ids, _ in results_by_thresh.values() for cid in ids]
    ))[:50]

    # rank table
    col_w = 70
    header = f'\n{"Rank":>4}  {"Candidate":{col_w}s}  {"Baseline":>8}'
    for t in THRESHOLDS:
        label = 'no-thr' if t == 0.0 else f't={t}'
        header += f'  {label:>8}'
    print(header)
    print('─' * (4 + 2 + col_w + 2 + 8 + len(THRESHOLDS) * 10))

    for i, cid in enumerate(all_ids, 1):
        short = cid[:col_w]
        is_relevant = '★' if cid in relevant else ' '
        b_r = str(baseline_ids.index(cid) + 1) if cid in baseline_ids else '-'
        row = f'{i:>4}{is_relevant} {short:{col_w}s}  {b_r:>8}'
        for t in THRESHOLDS:
            ids, _ = results_by_thresh[t]
            r = str(ids.index(cid) + 1) if cid in ids else '-'
            row += f'  {r:>8}'
        print(row)

    # recall@k summary
    print(f'\nRecall@k:')
    for k in [5, 10]:
        b_hit = any(r in baseline_ids[:k] for r in relevant)
        print(f'  k={k}  Baseline: {"HIT" if b_hit else "miss"}', end='')
        for t in THRESHOLDS:
            ids, n = results_by_thresh[t]
            hit = any(r in ids[:k] for r in relevant)
            label = 'no-thr' if t == 0.0 else f't={t}'
            print(f'  {label}: {"HIT" if hit else "miss"}(n={n})', end='')
        print()
    print()

════════════════════════════════════════════════════════════════════════════════
Query    : 10506_sent_27
Relevant : 040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140
Letter   : 10506  p467→p495  1534-11-01  degree=6
Letter   : 10506  p467→p495  1534-11-01  degree=6

Neighbour chunk similarity to query:
  file10471 (p467→p495, 1534-09-10)  sim=[0.589,0.792]  t=0.78: 4/192
  file10478 (p467→p495, 1534-09-25)  sim=[0.640,0.852]  t=0.78: 24/110
  file10501 (p495→p467, 1534-10-28)  sim=[0.623,0.821]  t=0.78: 20/136
  file10518 (?→p467, 1534-11-01)  sim=[0.629,0.859]  t=0.78: 131/302
  file10530 (p467→p495, 1534-11-01)  sim=[0.589,0.794]  t=0.78: 1/76

Rank  Candidate                                                               Baseline    no-thr    t=0.78
──────────────────────────────────────────────────────────────────────────────────────────────────────────
   1★ 040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140                 5         -         -
   2  025_

In [42]:
# --- Option 1: RRF over filtered neighbour chunk retrieval results ---
# Filter criterion: neighbour chunk's top-1 PSC score >= psc_threshold
# No new FAISS queries — reuses retrieval_results entirely

def enrich_metadata_rrf_existing(
    query_chunk_id: str,
    n_neighbours: int = N_NEIGHBOURS,
    k0: int = K0_RRF,
    psc_threshold: float = 0.78,
    max_chunks_per_neighbour: int = None   # optional hard cap per neighbour letter
) -> tuple[list[tuple[str, float]], int, int]:
    """
    Graph-neighbourhood RRF using existing baseline retrieval results.

    For each neighbour letter, selects chunks whose top-1 PSC candidate
    scores >= psc_threshold, takes their already-computed ranked lists,
    and fuses them with the query's own baseline list via RRF.

    Returns
    -------
    (fused_list, n_neighbour_letters_used, n_chunk_lists_used)
    """
    baseline = retrieval_results.get(query_chunk_id, [])
    query_letter_numeric = query_chunk_id.split('_')[0]
    graph_node_id = f'file{query_letter_numeric}'

    if graph_node_id not in G or G.degree(graph_node_id) == 0:
        return [(cid, score) for cid, score in baseline], 0, 0

    neighbours = sorted(G[graph_node_id].items(),
                        key=lambda x: x[1].get('date_min', ''))[:n_neighbours]

    all_proxy_lists = []
    n_letters_used = 0

    for nbr_node_id, _ in neighbours:
        nbr_letter_numeric = nbr_node_id.replace('file', '')
        chunk_indices = letter_to_chunk_indices.get(nbr_letter_numeric, [])
        if not chunk_indices:
            continue

        # get all chunk IDs for this neighbour letter
        nbr_chunk_ids = [bdc_ids[i] for i in chunk_indices]

        # filter: only chunks whose top-1 PSC score >= psc_threshold
        qualifying = []
        for cid in nbr_chunk_ids:
            result = retrieval_results.get(cid)
            if result and result[0][1] >= psc_threshold:
                qualifying.append((cid, result[0][1], result))

        if not qualifying:
            continue

        # optional: cap to top-N highest-confidence chunks per neighbour
        qualifying.sort(key=lambda x: -x[1])
        if max_chunks_per_neighbour:
            qualifying = qualifying[:max_chunks_per_neighbour]

        for _, _, ranked_list in qualifying:
            all_proxy_lists.append(ranked_list)

        n_letters_used += 1

    if not all_proxy_lists:
        return [(cid, score) for cid, score in baseline], 0, 0

    fused = reciprocal_rank_fusion([baseline] + all_proxy_lists, k0=k0)
    return fused, n_letters_used, len(all_proxy_lists)


print(f'retrieval_results covers {len(retrieval_results):,} chunks')

retrieval_results covers 961,431 chunks


In [60]:
# --- Smoke test: option 1 on GT queries ---

PSC_THRESHOLDS = [0.76, 0.78, 0.80, 0.82]

gt_test = {
    # letter 10404 — implicit, Augustine
    '10506_sent_27':     ['040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140'],
}

for sample_chunk, relevant in gt_test.items():
    if sample_chunk not in bdc_id_to_idx:
        print(f'{sample_chunk}: NOT IN EMBEDDING MATRIX — skipping\n')
        continue

    letter_id = sample_chunk.split('_')[0]
    nd = G.nodes.get(f'file{letter_id}', {})
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→'
          f'{nd.get("recipient_ref","?")}  {nd.get("date_str","?")}  '
          f'degree={G.degree(f"file{letter_id}")}')

    print(f'{"═"*80}')
    print(f'Query    : {sample_chunk}')
    print(f'Relevant : {relevant[0]}')
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→{nd.get("recipient_ref","?")}  '
          f'{nd.get("date_str","?")}  degree={G.degree(f"file{letter_id}")}')

    baseline_ids = [cid for cid, _ in retrieval_results.get(sample_chunk, [])[:10]]

    graph_node = f'file{letter_id}'
    if graph_node in G and G.degree(graph_node) > 0:
        neighbours = sorted(G[graph_node].items(),
                            key=lambda x: x[1].get('date_min', ''))[:N_NEIGHBOURS]
        print(f'\nQualifying neighbour chunks by PSC threshold:')
        for nbr_node, edata in neighbours:
            nbr_num = nbr_node.replace('file', '')
            chunk_ids = [bdc_ids[i] for i in letter_to_chunk_indices.get(nbr_num, [])]
            if not chunk_ids:
                continue
            nnd = G.nodes[nbr_node]
            print(f'  {nbr_node} ({nnd.get("sender_ref","?")}→{nnd.get("recipient_ref","?")}, '
                  f'{edata.get("date_min","?")})'
                  f'  sim=[{sims.min():.3f},{sims.max():.3f}]  {passing_str}')
            counts = {t: sum(
                1 for cid in chunk_ids
                if retrieval_results.get(cid) and retrieval_results[cid][0][1] >= t
            ) for t in PSC_THRESHOLDS}
            total = len(chunk_ids)
            count_str = '  '.join(f't={t}: {v}/{total}' for t, v in counts.items())
            print(f'  {nbr_node} ({nnd.get("sender_ref","?")}→{nnd.get("recipient_ref","?")}, '
                  f'{edata.get("date_min","?")})'
                  f'  sim=[{sims.min():.3f},{sims.max():.3f}]  {passing_str}')

    configs = [
        (0.78, None, 'opt1 t=0.78'),
        (0.80, None, 'opt1 t=0.80'),
        (0.78,    5, 'opt1 t=0.78 top5'),
        (0.80,    5, 'opt1 t=0.80 top5'),
    ]

    results_by_config = {}
    for t, mc, label in configs:
        fused, n_let, n_lists = enrich_metadata_rrf_existing(
            sample_chunk, psc_threshold=t, max_chunks_per_neighbour=mc
        )
        results_by_config[label] = ([cid for cid, _ in fused[:10]], n_let, n_lists)

    all_ids = list(dict.fromkeys(
        relevant + baseline_ids +
        [cid for ids, _, _ in results_by_config.values() for cid in ids]
    ))[:18]

    col_w = 56
    header = f'\n{"Rank":>4}  {"Candidate":{col_w}s}  {"Baseline":>8}'
    for _, _, label in configs:
        header += f'  {label:>16}'
    print(header)
    print('─' * (4 + 2 + col_w + 10 + len(configs) * 18))

    for i, cid in enumerate(all_ids, 1):
        mark = '★' if cid in relevant else ' '
        b_r = str(baseline_ids.index(cid) + 1) if cid in baseline_ids else '-'
        row = f'{i:>4}{mark} {cid[:col_w]:{col_w}s}  {b_r:>8}'
        for _, _, label in configs:
            ids, _, _ = results_by_config[label]
            r = str(ids.index(cid) + 1) if cid in ids else '-'
            row += f'  {r:>16}'
        print(row)

    print(f'\nRecall@k:')
    for k in [5, 10]:
        b_hit = any(r in baseline_ids[:k] for r in relevant)
        row = f'  k={k}  Baseline: {"HIT" if b_hit else "miss"}'
        for _, _, label in configs:
            ids, n_let, n_lists = results_by_config[label]
            hit = any(r in ids[:k] for r in relevant)
            row += f'  {label}: {"HIT" if hit else "miss"}(letters={n_let} lists={n_lists})'
        print(row)
    print()

Letter   : 10506  p467→p495  1534-11-01  degree=6
════════════════════════════════════════════════════════════════════════════════
Query    : 10506_sent_27
Relevant : 040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140
Letter   : 10506  p467→p495  1534-11-01  degree=6

Qualifying neighbour chunks by PSC threshold:
  file10471 (p467→p495, 1534-09-10)  sim=[0.589,0.794]  t=0.78: 1/76
  file10471 (p467→p495, 1534-09-10)  sim=[0.589,0.794]  t=0.78: 1/76
  file10478 (p467→p495, 1534-09-25)  sim=[0.589,0.794]  t=0.78: 1/76
  file10478 (p467→p495, 1534-09-25)  sim=[0.589,0.794]  t=0.78: 1/76
  file10501 (p495→p467, 1534-10-28)  sim=[0.589,0.794]  t=0.78: 1/76
  file10501 (p495→p467, 1534-10-28)  sim=[0.589,0.794]  t=0.78: 1/76
  file10518 (?→p467, 1534-11-01)  sim=[0.589,0.794]  t=0.78: 1/76
  file10518 (?→p467, 1534-11-01)  sim=[0.589,0.794]  t=0.78: 1/76
  file10530 (p467→p495, 1534-11-01)  sim=[0.589,0.794]  t=0.78: 1/76
  file10530 (p467→p495, 1534-11-01)  sim=[0.589,0.794]  t=

In [61]:
# --- Hybrid: query-similarity filter + existing retrieval RRF ---
# Filter: neighbour chunks similar to the query chunk (sim >= threshold)
# Signal: use their already-computed PSC retrieval results in RRF
# No new FAISS queries needed

def enrich_metadata_hybrid(
    query_chunk_id: str,
    n_neighbours: int = N_NEIGHBOURS,
    k0: int = K0_RRF,
    sim_threshold: float = 0.77,
    max_chunks_per_neighbour: int = None
) -> tuple[list[tuple[str, float]], int, int]:
    """
    For each neighbour letter, select chunks similar to the query chunk
    (sim >= sim_threshold), then use their existing PSC retrieval results
    in RRF fusion with the query's own baseline list.

    Returns
    -------
    (fused_list, n_neighbour_letters_used, n_chunk_lists_used)
    """
    baseline = retrieval_results.get(query_chunk_id, [])
    query_letter_numeric = query_chunk_id.split('_')[0]
    graph_node_id = f'file{query_letter_numeric}'

    if graph_node_id not in G or G.degree(graph_node_id) == 0:
        return [(cid, score) for cid, score in baseline], 0, 0

    query_idx = bdc_id_to_idx.get(query_chunk_id)
    if query_idx is None:
        return [(cid, score) for cid, score in baseline], 0, 0

    query_vec = bdc_embeddings[query_idx].astype('float32')

    neighbours = sorted(G[graph_node_id].items(),
                        key=lambda x: x[1].get('date_min', ''))[:n_neighbours]

    all_proxy_lists = []
    n_letters_used = 0

    for nbr_node_id, _ in neighbours:
        nbr_letter_numeric = nbr_node_id.replace('file', '')
        chunk_indices = letter_to_chunk_indices.get(nbr_letter_numeric, [])
        if not chunk_indices:
            continue

        # compute similarity of each neighbour chunk to the query chunk
        nbr_vecs = bdc_embeddings[chunk_indices].astype('float32')
        sims = nbr_vecs @ query_vec                    # (n_chunks,)

        # select chunks above threshold
        qualifying_indices = [
            chunk_indices[i] for i, s in enumerate(sims)
            if s >= sim_threshold
        ]

        if not qualifying_indices:
            continue

        # optionally cap to top-N most similar chunks per neighbour
        if max_chunks_per_neighbour:
            qualifying_indices = sorted(
                qualifying_indices,
                key=lambda i: -(bdc_embeddings[i] @ query_vec)
            )[:max_chunks_per_neighbour]

        # use existing PSC retrieval results for qualifying chunks
        letter_lists = []
        for idx in qualifying_indices:
            cid = bdc_ids[idx]
            result = retrieval_results.get(cid)
            if result:
                letter_lists.append(result)

        if not letter_lists:
            continue

        all_proxy_lists.extend(letter_lists)
        n_letters_used += 1

    if not all_proxy_lists:
        return [(cid, score) for cid, score in baseline], 0, 0

    fused = reciprocal_rank_fusion([baseline] + all_proxy_lists, k0=k0)
    return fused, n_letters_used, len(all_proxy_lists)


print('enrich_metadata_hybrid defined.')

enrich_metadata_hybrid defined.


In [66]:
# --- Smoke test: hybrid approach on GT queries ---

SIM_THRESHOLDS = [0.75, 0.77, 0.78, 0.80]

gt_test = {
    # letter 10404 — implicit, Augustine
    '10506_sent_27':     ['040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140'],
}

for sample_chunk, relevant in gt_test.items():
    if sample_chunk not in bdc_id_to_idx:
        print(f'{sample_chunk}: NOT IN EMBEDDING MATRIX — skipping\n')
        continue

    letter_id = sample_chunk.split('_')[0]
    nd = G.nodes.get(f'file{letter_id}', {})

    print(f'{"═"*80}')
    print(f'Query    : {sample_chunk}')
    print(f'Relevant : {relevant[0]}')
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→'
          f'{nd.get("recipient_ref","?")}  {nd.get("date_str","?")}  '
          f'degree={G.degree(f"file{letter_id}")}')

    query_idx = bdc_id_to_idx[sample_chunk]
    query_vec = bdc_embeddings[query_idx].astype('float32')
    baseline_ids = [cid for cid, _ in retrieval_results.get(sample_chunk, [])[:10]]

    # show qualifying counts per neighbour per threshold
    graph_node = f'file{letter_id}'
    if graph_node in G and G.degree(graph_node) > 0:
        neighbours = sorted(G[graph_node].items(),
                            key=lambda x: x[1].get('date_min', ''))[:N_NEIGHBOURS]
        print(f'\nQualifying neighbour chunks by query-similarity threshold:')
        for nbr_node, edata in neighbours:
            nbr_num = nbr_node.replace('file', '')
            indices = letter_to_chunk_indices.get(nbr_num, [])
            if not indices:
                continue
            vecs = bdc_embeddings[indices].astype('float32')
            sims = vecs @ query_vec
            counts = {t: int((sims >= t).sum()) for t in SIM_THRESHOLDS}
            nnd = G.nodes[nbr_node]
            count_str = '  '.join(f't={t}: {v}/{len(indices)}' for t, v in counts.items())
            print(f'  {nbr_node} ({nnd.get("sender_ref","?")}→{nnd.get("recipient_ref","?")}, '
                  f'{edata.get("date_min","?")})  {count_str}')

    # run all threshold variants
    configs = [
        (0.80, None, 't=0.80'),
        (0.80,    5, 't=0.80 top5'),
    ]

    results_by_config = {}
    for t, mc, label in configs:
        fused, n_let, n_lists = enrich_metadata_hybrid(
            sample_chunk, sim_threshold=t, max_chunks_per_neighbour=mc
        )
        results_by_config[label] = ([cid for cid, _ in fused[:10]], n_let, n_lists)

    all_ids = list(dict.fromkeys(
        relevant + baseline_ids +
        [cid for ids, _, _ in results_by_config.values() for cid in ids]
    ))[:50]

    col_w = 56
    header = f'\n{"Rank":>4}  {"Candidate":{col_w}s}  {"Baseline":>8}'
    for _, _, label in configs:
        header += f'  {label:>14}'
    print(header)
    print('─' * (4 + 2 + col_w + 10 + len(configs) * 16))

    for i, cid in enumerate(all_ids, 1):
        mark = '★' if cid in relevant else ' '
        b_r = str(baseline_ids.index(cid) + 1) if cid in baseline_ids else '-'
        row = f'{i:>4}{mark} {cid[:col_w]:{col_w}s}  {b_r:>8}'
        for _, _, label in configs:
            ids, _, _ = results_by_config[label]
            r = str(ids.index(cid) + 1) if cid in ids else '-'
            row += f'  {r:>14}'
        print(row)

    print(f'\nRecall@k:')
    for k in [5, 10]:
        b_hit = any(r in baseline_ids[:k] for r in relevant)
        row = f'  k={k}  Baseline: {"HIT" if b_hit else "miss"}'
        for _, _, label in configs:
            ids, n_let, n_lists = results_by_config[label]
            hit = any(r in ids[:k] for r in relevant)
            row += f'  {label}: {"HIT" if hit else "miss"}(l={n_let} n={n_lists})'
        print(row)
    print()

════════════════════════════════════════════════════════════════════════════════
Query    : 10506_sent_27
Relevant : 040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140
Letter   : 10506  p467→p495  1534-11-01  degree=6

Qualifying neighbour chunks by query-similarity threshold:
  file10471 (p467→p495, 1534-09-10)  t=0.75: 28/192  t=0.77: 7/192  t=0.78: 4/192  t=0.8: 0/192
  file10478 (p467→p495, 1534-09-25)  t=0.75: 68/110  t=0.77: 41/110  t=0.78: 24/110  t=0.8: 14/110
  file10501 (p495→p467, 1534-10-28)  t=0.75: 55/136  t=0.77: 30/136  t=0.78: 20/136  t=0.8: 7/136
  file10518 (?→p467, 1534-11-01)  t=0.75: 208/302  t=0.77: 157/302  t=0.78: 131/302  t=0.8: 59/302
  file10530 (p467→p495, 1534-11-01)  t=0.75: 11/76  t=0.77: 3/76  t=0.78: 1/76  t=0.8: 0/76

Rank  Candidate                                                 Baseline          t=0.80     t=0.80 top5
────────────────────────────────────────────────────────────────────────────────────────────────────────
   1★ 040_Augus

In [78]:
# --- Intersection-based re-ranking ---
# PSC candidates that appear in BOTH the query's and a neighbour chunk's
# retrieval results, scored by combined similarity.
# No new FAISS queries. No flooding — intersection narrows the candidate pool.

def enrich_metadata_intersection(
    query_chunk_id: str,
    n_neighbours: int = N_NEIGHBOURS,
    sim_threshold: float = 0.77,
    max_chunks_per_neighbour: int = None,
    combination: str = 'sum'   # 'sum', 'min', 'harmonic'
) -> tuple[list[tuple[str, float]], int, int]:
    """
    For each qualifying neighbour chunk (sim to query >= sim_threshold),
    find PSC candidates that appear in BOTH the query's retrieval list
    and the neighbour's retrieval list. Score by combined similarity.

    The intersection list is re-ranked by combined score and returned
    alongside the baseline list via RRF.

    Parameters
    ----------
    combination : how to combine query score + neighbour score
        'sum'      : score_q + score_n
        'min'      : min(score_q, score_n)  — conservative
        'harmonic' : 2*score_q*score_n / (score_q+score_n)

    Returns
    -------
    (fused_list, n_neighbour_letters_used, n_intersection_lists)
    """
    baseline = retrieval_results.get(query_chunk_id, [])
    query_letter_numeric = query_chunk_id.split('_')[0]
    graph_node_id = f'file{query_letter_numeric}'

    if graph_node_id not in G or G.degree(graph_node_id) == 0:
        return [(cid, score) for cid, score in baseline], 0, 0

    query_idx = bdc_id_to_idx.get(query_chunk_id)
    if query_idx is None:
        return [(cid, score) for cid, score in baseline], 0, 0

    query_vec = bdc_embeddings[query_idx].astype('float32')

    # query PSC results as dict for fast lookup
    query_psc = {cid: score for cid, score in baseline}

    neighbours = sorted(G[graph_node_id].items(),
                        key=lambda x: x[1].get('date_min', ''))[:n_neighbours]

    intersection_lists = []
    n_letters_used = 0

    for nbr_node_id, _ in neighbours:
        nbr_letter_numeric = nbr_node_id.replace('file', '')
        chunk_indices = letter_to_chunk_indices.get(nbr_letter_numeric, [])
        if not chunk_indices:
            continue

        # filter neighbour chunks by similarity to query
        nbr_vecs = bdc_embeddings[chunk_indices].astype('float32')
        sims = nbr_vecs @ query_vec

        qualifying = [
            (chunk_indices[i], float(sims[i]))
            for i in range(len(chunk_indices))
            if sims[i] >= sim_threshold
        ]
        if not qualifying:
            continue

        # optionally cap to top-N most similar
        qualifying.sort(key=lambda x: -x[1])
        if max_chunks_per_neighbour:
            qualifying = qualifying[:max_chunks_per_neighbour]

        letter_contributed = False
        for nbr_idx, _ in qualifying:
            nbr_chunk_id = bdc_ids[nbr_idx]
            nbr_psc = retrieval_results.get(nbr_chunk_id)
            if not nbr_psc:
                continue

            # intersection: PSC candidates in both query and neighbour lists
            nbr_psc_dict = {cid: score for cid, score in nbr_psc}
            common = set(query_psc.keys()) & set(nbr_psc_dict.keys())

            if not common:
                continue

            # score by combination function
            combined = []
            for psc_id in common:
                sq = query_psc[psc_id]
                sn = nbr_psc_dict[psc_id]
                if combination == 'sum':
                    score = sq + sn
                elif combination == 'min':
                    score = min(sq, sn)
                elif combination == 'harmonic':
                    score = 2 * sq * sn / (sq + sn) if (sq + sn) > 0 else 0
                else:
                    score = sq + sn
                combined.append((psc_id, score))

            combined.sort(key=lambda x: -x[1])
            intersection_lists.append(combined)
            letter_contributed = True

        if letter_contributed:
            n_letters_used += 1

    if not intersection_lists:
        return [(cid, score) for cid, score in baseline], 0, 0

    # fuse baseline + intersection lists via RRF
    fused = reciprocal_rank_fusion([baseline] + intersection_lists, k0=60)
    return fused, n_letters_used, len(intersection_lists)


print('enrich_metadata_intersection defined.')

enrich_metadata_intersection defined.


In [90]:
# --- Smoke test: intersection approach on GT queries ---

gt_test = {
    '10041_sent_211_213': ['016_Ambrosius-Mediolanensis_De-incarnationis-Dominicae-sacramento_window_34'],
    '10041_sent_232':     ['016_Ambrosius-Mediolanensis_De-incarnationis-Dominicae-sacramento_window_35'],
    '10041_sent_10':      ['016_Ambrosius-Mediolanensis_De-incarnationis-Dominicae-sacramento_window_8'],
    '10041_sent_220_222': ['002_Tertullianus_De-resurrectione-carnis_window_191'],
    '10404_sent_3_5':     ['076_Gregorius-I_Homiliae-in-Evangelia_window_706'],
    '10454_sent_18_20':   ['023_Hieronymus-Stridonensis_Dialogus-contra-Pelagianos_window_13'],
    '10478_sent_6_8':     ['033_Augustinus-Hipponensis_Epistolae_window_873'],
    '10506_sent_27':      ['040_Augustinus-Hipponensis_De-catechizandis-rudibus_window_140'],
}

CONFIGS = [
    (0.77, None, 'sum',      'for all t=0.77 sum'),
    (0.77,    3, 'sum',      'sum top3'),   # stricter cap
    (0.77,    1, 'sum',      'sum top1'),   # only best chunk per neighbour
]
for sample_chunk, relevant in gt_test.items():
    if sample_chunk not in bdc_id_to_idx:
        print(f'{sample_chunk}: NOT IN EMBEDDING MATRIX — skipping\n')
        continue

    letter_id = sample_chunk.split('_')[0]
    nd = G.nodes.get(f'file{letter_id}', {})
    baseline_ids = [cid for cid, _ in retrieval_results.get(sample_chunk, [])[:10]]

    print(f'{"═"*80}')
    print(f'Query    : {sample_chunk}')
    print(f'Relevant : {relevant[0]}')
    print(f'Letter   : {letter_id}  {nd.get("sender_ref","?")}→'
          f'{nd.get("recipient_ref","?")}  {nd.get("date_str","?")}  '
          f'degree={G.degree(f"file{letter_id}")}')

    # intersection size diagnostic per neighbour
    query_psc = {cid: score for cid, score in retrieval_results.get(sample_chunk, [])}
    query_idx = bdc_id_to_idx[sample_chunk]
    query_vec = bdc_embeddings[query_idx].astype('float32')

    graph_node = f'file{letter_id}'
    if graph_node in G and G.degree(graph_node) > 0:
        neighbours = sorted(G[graph_node].items(),
                            key=lambda x: x[1].get('date_min', ''))[:N_NEIGHBOURS]
        print(f'\nIntersection sizes with qualifying neighbour chunks (t=0.77):')
        for nbr_node, edata in neighbours:
            nbr_num = nbr_node.replace('file', '')
            indices = letter_to_chunk_indices.get(nbr_num, [])
            if not indices:
                continue
            vecs = bdc_embeddings[indices].astype('float32')
            sims = vecs @ query_vec
            qual_indices = [indices[i] for i, s in enumerate(sims) if s >= 0.77]
            if not qual_indices:
                print(f'  {nbr_node}: 0 qualifying chunks')
                continue
            # count intersections
            total_intersect = 0
            for idx in qual_indices:
                nbr_psc = {cid for cid, _ in retrieval_results.get(bdc_ids[idx], [])}
                total_intersect += len(set(query_psc.keys()) & nbr_psc)
            nnd = G.nodes[nbr_node]
            print(f'  {nbr_node} ({nnd.get("sender_ref","?")}→{nnd.get("recipient_ref","?")}, '
                  f'{edata.get("date_min","?")})  '
                  f'qual={len(qual_indices)}  total_intersections={total_intersect}')

    # run configs
    results_by_config = {}
    for t, mc, comb, label in CONFIGS:
        fused, n_let, n_lists = enrich_metadata_intersection(
            sample_chunk, sim_threshold=t,
            max_chunks_per_neighbour=mc, combination=comb
        )
        results_by_config[label] = ([cid for cid, _ in fused[:10]], n_let, n_lists)

    all_ids = list(dict.fromkeys(
        relevant + baseline_ids +
        [cid for ids, _, _ in results_by_config.values() for cid in ids]
    ))[:20]

    col_w = 70
    header = f'\n{"Rank":>4}  {"Candidate":{col_w}s}  {"Baseline":>8}'
    for _, _, _, label in CONFIGS:
        header += f'  {label:>16}'
    print(header)
    print('─' * (4 + 2 + col_w + 10 + len(CONFIGS) * 18))

    for i, cid in enumerate(all_ids, 1):
        mark = '★' if cid in relevant else ' '
        b_r = str(baseline_ids.index(cid) + 1) if cid in baseline_ids else '-'
        row = f'{i:>4}{mark} {cid[:col_w]:{col_w}s}  {b_r:>8}'
        for _, _, _, label in CONFIGS:
            ids, _, _ = results_by_config[label]
            r = str(ids.index(cid) + 1) if cid in ids else '-'
            row += f'  {r:>16}'
        print(row)

    print(f'\nRecall@k:')
    for k in [5, 10]:
        b_hit = any(r in baseline_ids[:k] for r in relevant)
        row = f'  k={k}  Baseline: {"HIT" if b_hit else "miss"}'
        for _, _, _, label in CONFIGS:
            ids, n_let, n_lists = results_by_config[label]
            hit = any(r in ids[:k] for r in relevant)
            row += f'  {label}: {"HIT" if hit else "miss"}(l={n_let} n={n_lists})'
        print(row)
    print()

════════════════════════════════════════════════════════════════════════════════
Query    : 10041_sent_211_213
Relevant : 016_Ambrosius-Mediolanensis_De-incarnationis-Dominicae-sacramento_window_34
Letter   : 10041  p495→p2832  1526-02-27  degree=2

Intersection sizes with qualifying neighbour chunks (t=0.77):
  file10038 (p495→p2832, 1526-01-21)  qual=10  total_intersections=4
  file10014 (p2832→p1043, 1526-01-31)  qual=47  total_intersections=13

Rank  Candidate                                                               Baseline  for all t=0.77 sum          sum top3          sum top1
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1★ 016_Ambrosius-Mediolanensis_De-incarnationis-Dominicae-sacramento_wind         1                 7                 3                 2
   2  002_Tertullianus_De-resurrectione-carnis_window_196                            2                 8                 4

In [71]:
# check if relevant appears in any neighbour intersection
print(f'\nRelevant candidate in intersections?')
for nbr_node, edata in neighbours:
    nbr_num = nbr_node.replace('file', '')
    indices = letter_to_chunk_indices.get(nbr_num, [])
    vecs = bdc_embeddings[indices].astype('float32')
    sims = vecs @ query_vec
    qual_indices = [indices[i] for i, s in enumerate(sims) if s >= 0.77]
    for idx in qual_indices:
        nbr_psc = dict(retrieval_results.get(bdc_ids[idx], []))
        if relevant[0] in nbr_psc:
            nbr_sim = float(bdc_embeddings[idx] @ query_vec)
            print(f'  FOUND in {bdc_ids[idx]} (sim={nbr_sim:.3f}) '
                  f'at PSC rank {list(nbr_psc.keys()).index(relevant[0])+1} '
                  f'score={nbr_psc[relevant[0]]:.4f}')


Relevant candidate in intersections?
  FOUND in 10518_sent_99 (sim=0.838) at PSC rank 6 score=0.8236
  FOUND in 10518_sent_49_51 (sim=0.821) at PSC rank 19 score=0.8334
  FOUND in 10518_sent_50_52 (sim=0.857) at PSC rank 19 score=0.8408
  FOUND in 10518_sent_51_53 (sim=0.829) at PSC rank 15 score=0.8412
  FOUND in 10518_sent_72_74 (sim=0.809) at PSC rank 16 score=0.8110
  FOUND in 10518_sent_90_92 (sim=0.786) at PSC rank 15 score=0.8352
  FOUND in 10518_sent_95_97 (sim=0.828) at PSC rank 8 score=0.8388


In [91]:
# --- Configuration ---
SIM_THRESHOLD        = 0.77
MAX_CHUNKS_PER_NBR   = 1
COMBINATION          = 'sum'

enrich_results = {}
fallback_ids   = []
n_lists_counts = []

print(f'Running Enrich-Metadata (intersection) over {len(vd_bdc_ids):,} BDC query chunks...')
print(f'  sim_threshold={SIM_THRESHOLD}, max_chunks_per_nbr={MAX_CHUNKS_PER_NBR}, '
      f'combination={COMBINATION}, sim_weight={SIM_WEIGHT}')

for qid in tqdm(vd_bdc_ids, desc='Enrich-Metadata'):
    fused, n_letters, n_lists = enrich_metadata_intersection(
        qid,
        sim_threshold=SIM_THRESHOLD,
        max_chunks_per_neighbour=MAX_CHUNKS_PER_NBR,
        combination=COMBINATION,
    )
    enrich_results[qid] = fused
    n_lists_counts.append(n_lists)
    if n_letters == 0:
        fallback_ids.append(qid)

print(f'\nDone.')
print(f'  Total queries              : {len(enrich_results):,}')
print(f'  Queries with graph support : {len(enrich_results) - len(fallback_ids):,}  '
      f'({(1 - len(fallback_ids)/len(enrich_results))*100:.1f}%)')
print(f'  Isolated fallbacks         : {len(fallback_ids):,}  '
      f'({len(fallback_ids)/len(enrich_results)*100:.1f}%)')
print(f'  Mean intersection lists    : {np.mean(n_lists_counts):.2f}')
print(f'  Max intersection lists     : {max(n_lists_counts)}')

Running Enrich-Metadata (intersection) over 19,466 BDC query chunks...
  sim_threshold=0.77, max_chunks_per_nbr=1, combination=sum, sim_weight=False


Enrich-Metadata:   0%|          | 0/19466 [00:00<?, ?it/s]


Done.
  Total queries              : 19,466
  Queries with graph support : 3,606  (18.5%)
  Isolated fallbacks         : 15,860  (81.5%)
  Mean intersection lists    : 0.28
  Max intersection lists     : 5


In [92]:
# save fused ranked lists from enrich_metadata_retrieve

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# Full fused results (all candidates returned by RRF, not truncated to K)
enrich_output = {
    "metadata": {
        "model_name"    : MODEL_NAME,
        "configuration" : "Enrich-Metadata",
        "n_neighbours"  : N_NEIGHBOURS,
        "k_proxy"       : K_PROXY,
        "k0_rrf"        : K0_RRF,
        "n_queries"     : len(enrich_results),
        "n_fallback"    : len(fallback_ids),
        "index_type"    : "FAISS IndexFlatIP",
    },
    "retrieval_results": {
        qid: [
            {"candidate_id": cid, "rrf_score": score, "rank": rank}
            for rank, (cid, score) in enumerate(ranked, 1)
        ]
        for qid, ranked in enrich_results.items()
    }
}

# JSON
json_path = output_dir / f"retrieval_results_{MODEL_NAME.lower()}_intersection__1_n{N_NEIGHBOURS}.json"
with open(json_path, 'w') as f:
    json.dump(enrich_output, f, indent=2)

# Pickle — stores enrich_results directly as {qid: [(cid, rrf_score), ...]}
# same structure as retrieval_results in the baseline notebook
pkl_path = output_dir / f"retrieval_results_{MODEL_NAME.lower()}_intersection_1_n{N_NEIGHBOURS}.pkl"
with open(pkl_path, 'wb') as f:
    pickle.dump(enrich_results, f)

print(f"Files saved:")
print(f"  JSON   : {json_path}")
print(f"  Pickle : {pkl_path}")

Files saved:
  JSON   : retrieval_results_e5-intertext_intersection__1_n5.json
  Pickle : retrieval_results_e5-intertext_intersection_1_n5.pkl
